# Autonomous Systems Portfolio 1  
***Connect 4 met Rule Based Systems***

![Connect4](https://upload.wikimedia.org/wikipedia/commons/a/ad/Connect_Four.gif)  
([Wikipedia, 4 Okt 2025](https://nl.wikipedia.org/wiki/Vier_op_%27n_rij))
|Name     |Studentnumber|Github    |
|---------|-------------|----------|
|Henry Lau|22122958     |HenryLau08|
|Michal|||
|Mohamed|||

## Inhoudsopgave



## 1. Introductie

### 1.1 Achtergrond  
#### 1.1.1. Connect 4
Connect 4 is een twee speler bordspel, waarbij spelers om de beurt een schrijf laten vallen in een kolom van een 7x6 boord. De speler die als eerste vier schijven op een rij krijgt (horizontaal, diagonaal of verticaal) wint. Wij hebben voor deze spel gekozen, omdat wij meer uitdaging wilden dan bij tic-tac-toe, maar tegelijkertijd niet gelijk in de diepe worden gegooid net als bij b.v. Mario.
Bij connect 4 zijn alle situaties op het bord in principe voorspelbaar en elke strategie wordt zo moeilijk als je het zelf maakt. Door het feit dat het raster een bepaald grootte heeft (7x6) en de bewegingen in principe beperkt zijn tot: winnen, blokkeren, positie innemen, leekt het ons een perfect spel om de rule-based systeem voor te bouwen.

#### 1.1.1. Doel van de opdracht
Het doel van dit project is om aantal rulebased strategieen te bouwen die zelfstandig beslissingen kunnen nemen tijdens het spel. Alhoewel winnen een leuke extra is, het gaat hierbij vooral om de basis van rule-based systeem te leren.

#### 1.1.2. Rule based systeem
Een rule based systeem is een systeem dat beslissingen maakt op basis van de vooraf ingestelde regels, die meestal in de vorm 'if-then' geschreven zijn.
Huidige situatie (genoemd state) wordt bijgehouden in working memory. Dit is een verzameling van alle kennis, die na elke stap wordt bijgewerkt.
Op basis van de huidige situatie voert het systeem een bepaald actie uit. ([Wikipedia, 27 Juli 2025](https://en.wikipedia.org/wiki/Rule-based_system)).  

Een voordeel van een rule-based systeem is dat het duidelijk is hoe keuzes worden gemaakt. Dankzij deze overzicht kan je relatief makkelijk achterhalen welke regel nu uitgevoerd is of waarom een actie genomen is. Om die reden is het ook makkelijk om je strategie aan te passen, je hoeft namelijk alleen de functies veranderen.

Een nadeel is dat alle regels van tevoren gemaakt zijn, wat dat betekent dat het systeem minder goed of helemaal niet om kan gaan met nieuwe of onverwachte situaties. Bovendien kan de strategie of jouw code heel onoverzichtelijk worden als de code te lang wordt of niet volgens de programmeerprincipes is geschreven, bijvoorbeeld DRY (Dont repeat yourself).
Tweede nadeel is conflicten. Het is mogelijk, vooral bij een uitgebreid systeem dat meerdere regels tegelijkertijd kunnen matchen. ([ScienceDirect, 2025](https://www.sciencedirect.com/topics/engineering/rule-based-system)).  

Rule-based systemen zijn nogsteeds handig in situaties waar het probleem duidelijk is en de regels makkelijk te bepalen zijn voor alle mogelijke situaties ([GeeksforGeeks, 6 Aug 2025](https://www.geeksforgeeks.org/machine-learning/rule-based-system-vs-machine-learning-system/)).

### Set Rules

#### Rule-based Set 1

Er wordt een vaste volgorde van regels gebruikt om actie te nemen op basis van de situatie en mogelijkheden.

Set 1:
1. Win
Als de agent kan winnen in deze beurt, dan kiest hij dat kolom.
2. Block Win
Als de tegenstander kan winnen in de volgende beurt, blokkeert de agent dit.
3. Block Fork
Als de tegenstander een fork kan maken, gaat het agent dit op tijd blokkeren.
4. Fork
Als de agent zelf een fork kan maken, doet hij dat.
5. Center
Als het midden vrij is, speelt de agent in het midden.
6. Random
Als geen van de bovenstaande regels geldt, dan speelt de agent willekeurig een beschikbare kolom.

De regels zijn geordend op basis van prioriteit:  
Als het agent kan winnen, dan speelt hij daar en eindigt het spel. Als tegenstander kan winnen of een fork kan maken op de volgende beurt, dan zal de agent dit blokkeren. Anders speelt het agent in het midden of willekeurig als midden niet meer beschikbaar is.

Een voorbeeld is als de agent op zijn beurt kan winnen en de tegenstander ook 3 schijven bij elkaar hebben, dan is de prioriteit om zijn eigen 4 te maken om te winnen en niet blokkeren.



In [1]:
import numpy as np
from pettingzoo.classic import connect_four_v3


# -----------------------------
# Initialize environment
# -----------------------------
env = connect_four_v3.env()
env.reset(seed=42)


# -----------------------------
# Convert Observation
# -----------------------------
def convert_observation(obs, agent):
    """
    Convert PettingZoo observation to absolute board:
    1 = player_0
    2 = player_1
    """
    board = np.zeros((6, 7), dtype=int)

    # channel 0 = current agent
    # channel 1 = opponent
    if agent == "player_0":
        me = 1
        opp = 2
    else:
        me = 2
        opp = 1

    for i in range(6):
        for j in range(7):
            if obs[i, j, 0] == 1:
                board[i, j] = me
            elif obs[i, j, 1] == 1:
                board[i, j] = opp

    return board


# -----------------------------
# Print Board
# -----------------------------
def print_board(board, mask):
    """
    Print het bord in de terminal.

    X = speler 1
    O = speler 2
    . = leeg

    mask laat zien welke kolommen nog geldig zijn.
    """
    print("\n" + "=" * 20)
    print("Current board:\n")

    for i in range(6):
        row = []
        for j in range(7):
            if board[i, j] == 1:
                row.append("X")
            elif board[i, j] == 2:
                row.append("O")
            else:
                row.append(".")
        print(" ".join(row))

    print("\nColumns:")
    print(" ".join([str(i) if mask[i] else "x" for i in range(7)]))
    print("=" * 20)

In [6]:
# -----------------------------
# Helpers
# -----------------------------
def drop_row(board, col):
    """
    Zoekt van onderaf de eerste lege cel in kolom 'col'.
    Geeft de rij-index terug waar de schijf zou landen.
    Kolom vol? Dan geeft het None terug.
    """
    for row in range(5, -1, -1):
        if board[row, col] == 0:
            return row
    return None

    
def drop_piece(board, col, player):
    """
    Laat een schijf vallen in kolom 'col'.

    Zoekt van onder naar boven een lege plek.
    Returnt nieuw bord.

    Als kolom vol is -> None
    """
    new_board = board.copy()

    for row in range(5, -1, -1):
        if new_board[row, col] == 0:
            new_board[row, col] = player
            return new_board

    return None


def check_win(board, player):
    r"""
    Checkt of 'player' 4 op een rij heeft.

    Richtingen:
    - horizontaal
    - verticaal
    - diagonaal (\)
    - diagonaal (/)

    Return: True / False
    """
    rows, cols = board.shape

    # Horizontal
    for r in range(rows):
        for c in range(cols - 3):
            if np.all(board[r, c:c+4] == player):
                return True

    # Vertical
    for c in range(cols):
        for r in range(rows - 3):
            if np.all(board[r:r+4, c] == player):
                return True

    # Diagonal (\)
    for r in range(rows - 3):
        for c in range(cols - 3):
            if all(board[r+i, c+i] == player for i in range(4)):
                return True

    # Diagonal (/)
    for r in range(3, rows):
        for c in range(cols - 3):
            if all(board[r-i, c+i] == player for i in range(4)):
                return True

    return False


def get_opponent(player):
    """
    Geeft de tegenstander terug.

    1 -> 2
    2 -> 1
    """
    return 2 if player == 1 else 1

def get_valid_moves(board):
    return np.array([board[0, c] == 0 for c in range(7)])

# -----------------------------
# Winning / Blocking
# -----------------------------
def find_winning_move(board, mask, player):
    """
    Zoekt een zet waarmee speler direct wint.

    Probeert alle geldige kolommen (mask).
    Simuleert zet en checkt winst.

    Return: kolom of None
    """
    for col in np.where(mask)[0]:
        temp = drop_piece(board, col, player)
        if temp is not None and check_win(temp, player):
            return col
    return None


def find_blocking_move(board, mask, player):
    """
    Blokkeert een winnende zet van de tegenstander.

    Gebruikt find_winning_move voor opponent.
    """
    return find_winning_move(board, mask, get_opponent(player))


# -----------------------------
# Fork Logic
# -----------------------------
def count_winning_moves(board, player):
    """
    Telt hoeveel winnende zetten speler heeft.

    Wordt gebruikt voor fork detectie.
    """
    mask = get_valid_moves(board)
    count = 0

    for col in np.where(mask)[0]:
        temp = drop_piece(board, col, player)
        if temp is not None and check_win(temp, player):
            count += 1

    return count


def find_fork_move(board, mask, player):
    """
    Zoekt een fork (2 winnende opties tegelijk).

    Simuleert zet en telt aantal wins daarna.
    >= 2 -> fork

    Return: kolom of None
    """
    for col in np.where(mask)[0]:
        temp = drop_piece(board, col, player)
        if temp is None:
            continue

        if count_winning_moves(temp, player) >= 2:
            return col
    return None


def find_blocking_fork(board, mask, player):
    """
    Blokkeert een fork van de tegenstander.
    """
    return find_fork_move(board, mask, get_opponent(player))


# -----------------------------
# Human Player
# -----------------------------
def get_human_action(observation):
    """
    Laat speler een zet kiezen via input.

    Checkt:
    - getal tussen 0-6
    - kolom niet vol

    Blijft vragen tot geldige input.
    """
    mask = observation["action_mask"]
    board = observation["board"]

    print_board(board, mask)

    while True:
        try:
            move = int(input("Choose column (0-6): "))
            if 0 <= move < 7 and mask[move]:
                return move
            print("Invalid move, try again.")
        except:
            print("Enter a valid number.")


# -----------------------------
# AI Strategy
# -----------------------------
def get_current_player(board):
    # tel aantal stukken
    count1 = np.sum(board == 1)
    count2 = np.sum(board == 2)

    # speler met minder zetten is aan de beurt
    return 1 if count1 == count2 else 2

def smart_strategy(observation, agent):
    """
    Rule-based AI strategie.

    Prioriteit:
    1. Win
    2. Block win
    3. Block fork
    4. Fork
    5. Center (kolom 3)
    6. Random

    Return: gekozen kolom
    """
    board = observation["board"]
    mask = observation["action_mask"]

    player = get_current_player(board)

    print("Agent:", agent)
    print("Detected player:", player)

    # 1. Win
    move = find_winning_move(board, mask, player)
    if move is not None:
        return int(move)

    # 2. Block
    move = find_blocking_move(board, mask, player)
    if move is not None:
        return int(move)

    # 3. Block fork
    move = find_blocking_fork(board, mask, player)
    if move is not None:
        return int(move)

    # 4. Fork
    move = find_fork_move(board, mask, player)
    if move is not None:
        return int(move)

    # 5. Center
    valid = np.where(mask)[0]
    if 3 in valid:
        return 3

    # 6. Random
    return int(np.random.choice(valid))

Michal Strategie

In [ ]:
def defensive_strategy(observation, agent):
    board = observation["board"]
    mask  = observation["action_mask"]
    valid_cols = [int(c) for c in np.where(mask)[0]]

    if not valid_cols:
        return 0

    player = 1 if agent == "player_0" else 2
    opp    = get_opponent(player)

    def gives_free_win_above(col):
        row = drop_row(board, col)
        if row is None or row == 0:
            return False
        sim = drop_piece(board, col, player)
        if sim is None:
            return False
        test = sim.copy()
        test[row - 1, col] = opp
        return check_win(test, opp)

    def future_opp_wins(col, mover=None):
        if mover is None:
            mover = player
        sim = drop_piece(board, col, mover)
        if sim is None:
            return 999
        opp_mask = get_valid_moves(sim)
        return count_winning_moves(sim, opp) if opp_mask.any() else 0

    def find_fork_cols(p):
        return [
            col for col in valid_cols
            if (drop_piece(board, col, p) is not None
                and count_winning_moves(drop_piece(board, col, p), p) >= 2)
        ]

    def future_my_forks(col):
        sim = drop_piece(board, col, player)
        if sim is None:
            return 0
        sim_mask = get_valid_moves(sim)
        return sum(
            1 for c in np.where(sim_mask)[0]
            if drop_piece(sim, c, player) is not None
            and count_winning_moves(drop_piece(sim, c, player), player) >= 2
        )

    # Win
    win = find_winning_move(board, mask, player)
    if win is not None:
        return int(win)

    # Blok win
    block = find_blocking_move(board, mask, player)
    if block is not None:
        return int(block)

    # Fork
    my_forks = find_fork_cols(player)
    if my_forks:
        safe = [c for c in my_forks if not gives_free_win_above(c)]
        pool = safe if safe else my_forks
        return min(pool, key=future_opp_wins)

    # naar fork bouwen
    safe_cols = [c for c in valid_cols if not gives_free_win_above(c)]
    build_pool = safe_cols if safe_cols else valid_cols
    best_build = max(build_pool, key=future_my_forks)
    if future_my_forks(best_build) > 0:
        return best_build

    # Blokkeer fork
    opp_forks = find_fork_cols(opp)
    if opp_forks:
        safe_cols = [c for c in valid_cols if not gives_free_win_above(c)]
        pool = safe_cols if safe_cols else valid_cols
        return min(pool, key=future_opp_wins)

    # Valstrik vermijden, midden, order
    safe_cols = [c for c in valid_cols if not gives_free_win_above(c)]
    pool = safe_cols if safe_cols else valid_cols

    if 3 in pool:
        return 3

    for col in [2, 4, 1, 5, 0, 6]:
        if col in pool:
            return col

    return pool[0]

In [3]:
# -----------------------------
# Game Loop
# -----------------------------
game_over = False

for agent in env.agent_iter():
    obs, reward, term, trunc, _ = env.last()

    # Dead agent -> must pass None
    if term or trunc:
      if not game_over:
          game_over = True

          # Print final board
          final_board = convert_observation(obs["observation"], agent)
          print("\nFinal position:")
          print_board(final_board, obs["action_mask"])

          # Print result
          if reward == 1:
              print("\nYou win!" if agent == "player_0" else "\nComputer wins!")
          elif reward == 0:
              print("\nDraw!")

    else:
        # Alive agent -> compute action
        board = convert_observation(obs["observation"], agent)
        observation = {
            "board": board,
            "action_mask": obs["action_mask"]
        }

        if agent == "player_0":
            action = get_human_action(observation)
        else:
            action = smart_strategy(observation, agent)

    env.step(action)

env.close()


Current board:

. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .

Columns:
0 1 2 3 4 5 6
Agent: player_1
Detected player: 2

Current board:

. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. . . O . X .

Columns:
0 1 2 3 4 5 6
Agent: player_1
Detected player: 2

Current board:

. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. . . O . X .
. . . O . X .

Columns:
0 1 2 3 4 5 6
Agent: player_1
Detected player: 2

Current board:

. . . . . . .
. . . . . . .
. . . . . . .
. . . O . . .
. . . O . X .
. . . O . X X

Columns:
0 1 2 3 4 5 6
Agent: player_1
Detected player: 2

Final position:

Current board:

. . . . . . .
. . . . . . .
. . . O . . .
. . . O . X .
. . . O . X .
. . . O . X X

Columns:
0 1 2 3 4 5 6


ValueError: when an agent is dead, the only valid action is None

## Bronnen

- Wikipedia, 4 Okt 2025. Vier op 'n rij. https://nl.wikipedia.org/wiki/Vier_op_%27n_rij
- Wikipedia, 27 Juli 2025. Rule-based system. https://en.wikipedia.org/wiki/Rule-based_system
- GeeksforGeeks, 6 Aug 2025. Rule Based System Vs Machine Learning System. https://www.geeksforgeeks.org/machine-learning/rule-based-system-vs-machine-learning-system/

